In [1]:
from pyspark.sql import SparkSession  
import getpass 
username = getpass 
spark = SparkSession.builder.config('spark.ui.port','0').\
        config('spark.sql.warehouse.dir',f"/user/itv022063/warehouse").\
        enableHiveSupport().\
        master('yarn').\
        getOrCreate()

In [2]:
defaulter_schema = """member_id string, delinq_2yrs float, delinq_amnt float,
pub_rec float, pub_rec_bankruptcies float, inq_last_6mths float,
total_rec_late_fee float, mths_since_last_delinq float, mths_since_last_record float"""

In [3]:
loandf=spark.read.format('csv')\
.schema(defaulter_schema)\
.option("header","true") \
.load("/user/itv022063/Yash/raw/Loan_Defaulters")

In [4]:
loandf.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- delinq_2yrs: float (nullable = true)
 |-- delinq_amnt: float (nullable = true)
 |-- pub_rec: float (nullable = true)
 |-- pub_rec_bankruptcies: float (nullable = true)
 |-- inq_last_6mths: float (nullable = true)
 |-- total_rec_late_fee: float (nullable = true)
 |-- mths_since_last_delinq: float (nullable = true)
 |-- mths_since_last_record: float (nullable = true)



In [5]:
from pyspark.sql.functions import *;
loans_def_processed_df = loandf.withColumn("delinq_2yrs", col("delinq_2yrs").cast("integer")).fillna(0, subset = ["delinq_2yrs"])

In [6]:
loans_def_processed_df.filter(col("delinq_2yrs").isNull()).count()

0

In [7]:
loans_def_processed_df \
    .groupBy("delinq_2yrs") \
    .agg(count("*").alias("total")) \
    .orderBy(col("total").desc()) \
    .show(40)

+-----------+-------+
|delinq_2yrs|  total|
+-----------+-------+
|          0|1839141|
|          1| 281337|
|          2|  81285|
|          3|  29545|
|          4|  13180|
|          5|   6601|
|          6|   3719|
|          7|   2063|
|          8|   1226|
|          9|    821|
|         10|    558|
|         11|    363|
|         12|    266|
|         13|    167|
|         14|    123|
|         15|     90|
|         16|     56|
|         17|     33|
|         18|     32|
|         19|     24|
|         20|     19|
|         21|     16|
|         22|      7|
|         24|      6|
|         23|      5|
|         26|      4|
|         29|      2|
|         25|      2|
|         30|      2|
|         28|      1|
|         27|      1|
|         32|      1|
|         35|      1|
|         39|      1|
|         58|      1|
|         42|      1|
|         36|      1|
+-----------+-------+



In [8]:
loans_def_delinq_df = (
    loans_def_processed_df
    .select(
        col("member_id"),
        col("delinq_2yrs"),
        col("delinq_amnt"),
        col("mths_since_last_delinq").cast("int").alias("mths_since_last_delinq")
    )
    .filter(
        (col("delinq_2yrs") > 0) |
        (col("mths_since_last_delinq") > 0)
    )
)

In [9]:
loans_def_records_enq_df = (
    loans_def_processed_df
    .select(
        col("member_id")
    ).filter(
        (col("pub_rec") > 0.0) | 
        (col("pub_rec_bankruptcies") > 0.0) | 
        (col("inq_last_6mths") > 0.0)
    )
)

In [10]:
loans_def_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loans_def_delinq_parquet") \
.save()

In [11]:
loans_def_delinq_df.write \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loans_def_delinq_csv") \
.save()

In [12]:
loans_def_records_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loans_def_records_enq_parquet") \
.save()

In [13]:
loans_def_delinq_df.write \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loans_def_records_enq_csv") \
.save()

In [14]:
loans_pub_rec=loans_def_processed_df.withColumn("pub_rec", col("pub_rec").cast("integer")).fillna(0, subset = ["pub_rec"])

In [15]:
loans_pub_rec_bankruptcies=loans_pub_rec.withColumn("pub_rec_bankruptcies", col("pub_rec_bankruptcies").cast("integer")).fillna(0, subset = ["pub_rec_bankruptcies"])

In [16]:
loans_inq_last_6mths=loans_pub_rec_bankruptcies.withColumn("inq_last_6mths", col("inq_last_6mths").cast("integer")).fillna(0, subset = ["inq_last_6mths"])

In [17]:
loans_inq_last_6mths

member_id,delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee,mths_since_last_delinq,mths_since_last_record
b59d80da191f5b573...,0,0.0,0,0,1,0.0,31.0,null
202d9f56ecb7c3bc9...,1,0.0,0,0,0,0.0,6.0,null
e5a140c0922b554b9...,0,0.0,0,0,0,0.0,47.0,null
e12aefc548f750777...,0,0.0,0,0,0,0.0,33.0,null
1b3a50d854fbbf97e...,1,0.0,0,0,0,0.0,21.0,null
1c4329e5f17697127...,0,0.0,0,0,0,0.0,null,null
5026c86ad983175eb...,0,0.0,1,0,2,0.0,null,71.0
9847d8c1e9d0b2084...,1,0.0,2,0,0,0.0,6.0,63.0
8340dbe1adea41fb4...,0,0.0,0,0,0,0.0,36.0,null
d4de0de3ab7d79ad4...,0,0.0,0,0,0,0.0,35.0,null


In [18]:
loans_def_detail_records_enq_df = loans_inq_last_6mths.select("member_id", "pub_rec", "pub_rec_bankruptcies", "inq_last_6mths")

In [19]:
loans_def_detail_records_enq_df

member_id,pub_rec,pub_rec_bankruptcies,inq_last_6mths
b59d80da191f5b573...,0,0,1
202d9f56ecb7c3bc9...,0,0,0
e5a140c0922b554b9...,0,0,0
e12aefc548f750777...,0,0,0
1b3a50d854fbbf97e...,0,0,0
1c4329e5f17697127...,0,0,0
5026c86ad983175eb...,1,0,2
9847d8c1e9d0b2084...,2,0,0
8340dbe1adea41fb4...,0,0,0
d4de0de3ab7d79ad4...,0,0,0


In [20]:
loans_def_detail_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loans_def_detail_records_enq_df_csv") \
.save()

In [21]:
loans_def_detail_records_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loans_def_detail_records_enq_df_parquet") \
.save()

In [22]:
spark.stop()